In [ ]:
# Lab type: review
# Course: AI-Assisted Data Science
# Lesson: Capstone — End-to-End Audit of an AI-Generated Analysis Notebook
# Task: Find and document five problems in a runnable AI-generated churn pipeline

# Capstone Lab: Auditing an AI-Generated Churn Pipeline

An AI tool was given this prompt:

> *Analyse customer churn data and build a model to predict which customers are likely to leave.*

The generated pipeline runs without errors and reports **99.5% accuracy**.
It contains **five distinct problems** — some subtle, some serious.
Your job is to find them all.

Work through this notebook in order:
1. Run the data setup and the AI-generated pipeline
2. Use the audit sections to detect each problem
3. Document each finding in the provided format
4. Implement and run the corrected pipeline

---
## Part 1: Data setup

The dataset contains 2,000 customer records with these columns:

| Column | Type | Notes |
|---|---|---|
| `customer_id` | str | Unique identifier |
| `tenure_months` | int | Months as a customer |
| `monthly_spend` | float | Avg monthly spend; ~6% missing |
| `support_tickets` | int | Support contacts in last 90 days |
| `last_login_days_ago` | int | Days since last login |
| `contract_type` | str | `'monthly'`, `'annual'`, `'two-year'` |
| `region` | str | Geographic region; ~1% missing |
| `churn_date` | str | Date churned; null for active customers |
| `churned` | int | Target: 1 = churned, 0 = active |

In [ ]:
import pandas as pd
import numpy as np

rng = np.random.default_rng(17)
n = 2000

tenure    = rng.integers(1, 73, n).astype(float)
spend     = rng.normal(50, 20, n).round(2)
tickets   = rng.integers(0, 15, n)
login_days = rng.integers(0, 365, n)
contract  = rng.choice(["monthly", "annual", "two-year"], n, p=[0.5, 0.3, 0.2])
region    = rng.choice(["North", "South", "East", "West"], n).tolist()

# Churn probability driven by tickets, recency, tenure, and contract type
contract_arr  = np.array(contract)
contract_risk = np.where(contract_arr == "monthly", 1.0,
                np.where(contract_arr == "annual",  0.0, -0.5))
logit      = -3.0 + 0.25 * tickets + 0.006 * login_days - 0.03 * tenure + contract_risk
churn_prob = 1 / (1 + np.exp(-logit))
churned    = (rng.random(n) < churn_prob).astype(int)

# churn_date: set only for customers who churned
base_date   = pd.Timestamp("2024-01-01")
review_date = pd.Timestamp("2024-06-01")   # fixed analysis date for reproducibility
churn_dates = []
for i in range(n):
    if churned[i] == 1:
        days_back = int(rng.integers(30, 730))
        churn_dates.append((base_date - pd.Timedelta(days=days_back)).strftime("%Y-%m-%d"))
    else:
        churn_dates.append(None)

# Inject missing values
spend_list = spend.tolist()
for idx in rng.choice(n, int(n * 0.06), replace=False):
    spend_list[idx] = None
for idx in rng.choice(n, int(n * 0.01), replace=False):
    region[idx] = None

df_original = pd.DataFrame({
    "customer_id":        [f"C{i:05d}" for i in range(n)],
    "tenure_months":      tenure.tolist(),
    "monthly_spend":      spend_list,
    "support_tickets":    tickets.tolist(),
    "last_login_days_ago": login_days.tolist(),
    "contract_type":      contract.tolist(),
    "region":             region,
    "churn_date":         churn_dates,
    "churned":            churned.tolist(),
})

print(f"Dataset: {df_original.shape}")
print(f"Churn rate: {df_original['churned'].mean():.1%}")
print(f"\nMissing values:")
print(df_original.isna().sum()[df_original.isna().sum() > 0])
df_original.head()

---
## Part 2: The AI-generated pipeline

Run the cell below. It is the pipeline exactly as the AI tool produced it.
**Read it before you run it.** Note every decision the AI made that you didn't specify.

In [ ]:
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

df = df_original.copy()

# --- Feature engineering ---
df["days_since_churn"] = (review_date - pd.to_datetime(df["churn_date"])).dt.days
df["spend_per_tenure"] = df["monthly_spend"].astype(float) / df["tenure_months"]

# --- Preprocessing ---
df = df.fillna(0)

le = LabelEncoder()
for col in df.select_dtypes("object").columns:
    df[col] = le.fit_transform(df[col].astype(str))

X = df.drop("churned", axis=1)
y = df["churned"]

scaler = StandardScaler()
numeric_cols = X.select_dtypes("number").columns
X[numeric_cols] = scaler.fit_transform(X[numeric_cols])

# --- Split ---
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# --- Model ---
model_broken = RandomForestClassifier(n_estimators=100, random_state=42)
model_broken.fit(X_train, y_train)

broken_accuracy = accuracy_score(y_test, model_broken.predict(X_test))
print(f"Accuracy: {broken_accuracy:.1%}")

In [ ]:
# What is the model actually relying on?
importances = pd.Series(
    model_broken.feature_importances_,
    index=X.columns
).sort_values(ascending=False)

print("Top 5 features by importance:")
print(importances.head().round(3))

> **Before you continue:** Two columns account for nearly all the model's predictive power.
> Both of their names contain the word *churn*. Write down why that is suspicious.

*(Write your initial observation here.)*

---
## Part 3: Audit — five findings

Work through each finding in order. For each one:
1. Run the detection cells
2. Answer the question
3. Complete the finding summary at the end of each section

The finding summary format is:
- **What the code does:**
- **Why it is wrong:**
- **Correct approach:**

### Finding 1: Target leakage

*Hint: Look at `days_since_churn`. What is `churn_date` for active customers?
What does subtracting `NaT` from a date produce? And after `fillna(0)`, what value
does an active customer have in this column?*

In [ ]:
# Trace days_since_churn for active vs churned customers
probe = df_original[["churn_date", "churned"]].copy()
probe["days_since_churn_raw"] = (review_date - pd.to_datetime(probe["churn_date"])).dt.days
probe["days_since_churn_filled"] = probe["days_since_churn_raw"].fillna(0)

print("Active customers (churned=0):")
print(probe[probe["churned"] == 0][["churn_date", "days_since_churn_raw", "days_since_churn_filled"]].head(5))

print("\nChurned customers (churned=1):")
print(probe[probe["churned"] == 1][["churn_date", "days_since_churn_raw", "days_since_churn_filled"]].head(5))

In [ ]:
# Confirm days_since_churn perfectly separates the classes
print("days_since_churn_filled grouped by churned:")
print(probe.groupby("churned")["days_since_churn_filled"].describe().round(1))

# churn_date itself also remains in X — show how it encodes the target after LabelEncoder
le_probe = LabelEncoder()
probe["churn_date_encoded"] = le_probe.fit_transform(probe["churn_date"].fillna(0).astype(str))
print("\nchurn_date_encoded grouped by churned (0 = active):")
print(probe.groupby("churned")["churn_date_encoded"].describe().round(1))

> **Question:** At inference time — when you use this model on active customers
> to predict who will churn next month — what value will `days_since_churn` have
> for every active customer? What will the model predict for all of them?

*(Write your answer here.)*

**Finding 1 summary** *(complete this)*

- **What the code does:**
- **Why it is wrong:**
- **Correct approach:**

### Finding 2: Division by zero in feature engineering

*Hint: Look at `spend_per_tenure`. What happens if `tenure_months` is 0?
New customers on their first day have `tenure_months == 0`.*

In [ ]:
# Does this dataset happen to contain any tenure_months == 0?
print(f"tenure_months == 0 in this dataset: {(df_original['tenure_months'] == 0).sum()}")
print(f"Min tenure_months: {df_original['tenure_months'].min()}")

# The bug is latent — simulate what would happen with one new customer row
new_customer = pd.DataFrame([{
    "customer_id": "C99999",
    "tenure_months": 0.0,
    "monthly_spend": 45.00,
    "support_tickets": 0,
    "last_login_days_ago": 1,
    "contract_type": "monthly",
    "region": "North",
    "churn_date": None,
    "churned": 0,
}])

new_customer["spend_per_tenure"] = new_customer["monthly_spend"] / new_customer["tenure_months"]
print(f"\nspend_per_tenure for new customer (tenure=0): {new_customer['spend_per_tenure'].values[0]}")
print(f"Type: {type(new_customer['spend_per_tenure'].values[0])}")

In [ ]:
# Show what happens downstream: fillna(0) does NOT fix inf values
import math
inf_series = pd.Series([float("inf"), 45.0, 60.0, float("nan")])
print("Before fillna(0):", inf_series.tolist())
print("After  fillna(0):", inf_series.fillna(0).tolist())
print("\nInf remains. StandardScaler on a column containing inf produces NaN.")
print("RandomForest raises ValueError on NaN features.")

> **Question:** This bug doesn't affect the current dataset (minimum tenure is 1).
> Why is it still a problem that needs fixing before this pipeline goes to production?

*(Write your answer here.)*

**Finding 2 summary** *(complete this)*

- **What the code does:**
- **Why it is wrong:**
- **Correct approach:**

### Finding 3: `fillna(0)` applied to all columns

*Hint: Which columns have missing values? Is zero a meaningful fill value for all of them?*

In [ ]:
# Show the effect of fillna(0) on each column that had missing values
df_before = df_original.copy()
df_after  = df_original.fillna(0)

for col in ["monthly_spend", "region"]:
    missing = df_before[col].isna().sum()
    print(f"--- {col} ({missing} missing) ---")
    print(f"  dtype before: {df_before[col].dtype}")
    print(f"  dtype after:  {df_after[col].dtype}")
    print(f"  unique values after (sample): {df_after[col].unique()[:8]}")
    print()

In [ ]:
# The '0' string category is now in region — it will be label-encoded as a real region
region_filled = df_after["region"].astype(str)
print("region value_counts after fillna(0):")
print(region_filled.value_counts())
print()

# monthly_spend: how many rows now have spend == 0?
zero_spend = (pd.to_numeric(df_after["monthly_spend"], errors="coerce") == 0).sum()
print(f"Rows with monthly_spend == 0 after fillna: {zero_spend}")
print(f"(All of these are imputed — no real £0 spenders exist in the dataset.)")

> **Question:** If a model learns that customers with `region == '0'` behave differently,
> what will happen at inference time when a new row arrives with a missing region?

*(Write your answer here.)*

**Finding 3 summary** *(complete this)*

- **What the code does:**
- **Why it is wrong:**
- **Correct approach:**

### Finding 4: LabelEncoder on `contract_type`

*Hint: `contract_type` has a natural order: monthly < annual < two-year.
What numeric order does `LabelEncoder` assign? What happens if the test set
contains a contract type the encoder hasn't seen?*

In [ ]:
# LabelEncoder assigns alphabetical order — not the meaningful contract order
le_contract = LabelEncoder()
le_contract.fit(["monthly", "annual", "two-year"])

print("LabelEncoder encoding (alphabetical):")
for label, code in zip(le_contract.classes_, range(len(le_contract.classes_))):
    print(f"  '{label}' → {code}")

print()
print("Expected meaningful order:")
for i, label in enumerate(["monthly", "annual", "two-year"]):
    print(f"  '{label}' → {i}  (commitment level low → high)")

In [ ]:
# The loop reuses the same `le` object — after the loop, `le.classes_` holds
# only the classes from the LAST column processed.
# This means you cannot use `le` to inverse-transform or infer on contract_type.
df_check = df_original.fillna(0).copy()
le_loop = LabelEncoder()
for col in df_check.select_dtypes("object").columns:
    df_check[col] = le_loop.fit_transform(df_check[col].astype(str))

print("le.classes_ after the loop (only the LAST column's classes):")
print(le_loop.classes_[:10], "...")

print("\nIf a new contract type 'enterprise' appeared at inference time:")
try:
    le_contract.transform(["enterprise"])
except ValueError as e:
    print(f"  ValueError: {e}")

> **Question:** `LabelEncoder` assigns `'annual' → 0`, `'monthly' → 1`, `'two-year' → 2`.
> A linear model would interpret this as: annual is the lowest, two-year is the highest commitment.
> But `monthly` sits between them at 1. Why does this corrupt the model's understanding
> of the ordinal relationship in contract type?

*(Write your answer here.)*

**Finding 4 summary** *(complete this)*

- **What the code does:**
- **Why it is wrong:**
- **Correct approach:**

### Finding 5: Scaler fit on the full feature matrix before the train/test split

*Hint: `scaler.fit_transform(X[numeric_cols])` is called on the full X — before
`train_test_split`. What data does the scaler see when it fits?*

In [ ]:
# Demonstrate the contamination: the scaler's learned mean should match
# the TRAINING set mean, not the full dataset mean

# Rebuild a clean (unscaled) version of X to test this
df_probe = df_original.copy()
df_probe["days_since_churn"] = (review_date - pd.to_datetime(df_probe["churn_date"])).dt.days
df_probe["spend_per_tenure"] = df_probe["monthly_spend"].astype(float) / df_probe["tenure_months"]
df_probe = df_probe.fillna(0)
le_p = LabelEncoder()
for col in df_probe.select_dtypes("object").columns:
    df_probe[col] = le_p.fit_transform(df_probe[col].astype(str))
X_probe = df_probe.drop("churned", axis=1)
y_probe = df_probe["churned"]

# Train/test split indices (same seed as broken pipeline)
from sklearn.model_selection import train_test_split
idx_train, idx_test = train_test_split(range(len(X_probe)), test_size=0.2, random_state=42)

col = "last_login_days_ago"   # a non-leaky numeric column

full_mean  = X_probe[col].mean()
train_mean = X_probe[col].iloc[idx_train].mean()
test_mean  = X_probe[col].iloc[idx_test].mean()

print(f"'{col}' mean — full dataset:   {full_mean:.2f}")
print(f"'{col}' mean — training set:   {train_mean:.2f}")
print(f"'{col}' mean — test set:       {test_mean:.2f}")

from sklearn.preprocessing import StandardScaler
sc = StandardScaler()
sc.fit(X_probe[[col]])
print(f"\nScaler learned mean (fit on full X): {sc.mean_[0]:.2f}")
print(f"Expected scaler mean if fit on train only: {train_mean:.2f}")
print(f"Difference: {abs(sc.mean_[0] - train_mean):.2f}")

> **Question:** The difference between the contaminated mean and the train-only mean
> is small for this feature. Why is data leakage through the scaler still a serious problem,
> even when the magnitude is small?

*(Write your answer here.)*

**Finding 5 summary** *(complete this)*

- **What the code does:**
- **Why it is wrong:**
- **Correct approach:**

---
## Part 4: Corrected pipeline

The corrected pipeline addresses all five findings:

| Finding | Fix |
|---|---|
| 1. Target leakage | Drop `days_since_churn`, `churn_date`, and `customer_id` before modelling |
| 2. Division by zero | Guard `tenure_months` with `.replace(0, np.nan)` before dividing |
| 3. `fillna(0)` | Use per-column imputation: median for numeric, most-frequent for categorical |
| 4. LabelEncoder | Use `OrdinalEncoder` with explicit category order for `contract_type`; one-hot for `region` |
| 5. Scaler before split | Split first; fit all transformers on training data only using `Pipeline` + `ColumnTransformer` |

In [ ]:
from sklearn.preprocessing import StandardScaler, OrdinalEncoder, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

df_c = df_original.copy()

# Finding 1 + 2: no leaky feature; guard division by zero
df_c["spend_per_tenure"] = (
    df_c["monthly_spend"].astype(float)
    / df_c["tenure_months"].replace(0, np.nan)
)

# Finding 1: drop identifier and post-event columns
df_c = df_c.drop(columns=["customer_id", "churn_date"])

X_c = df_c.drop("churned", axis=1)
y_c = df_c["churned"]

# Finding 5: split BEFORE any fitting
X_train, X_test, y_train, y_test = train_test_split(
    X_c, y_c, test_size=0.2, random_state=42, stratify=y_c
)

numeric_cols   = ["tenure_months", "monthly_spend", "support_tickets",
                  "last_login_days_ago", "spend_per_tenure"]
ordinal_cols   = ["contract_type"]   # has a meaningful order
nominal_cols   = ["region"]          # no meaningful order

# Finding 3 + 4 + 5: per-column imputation, explicit ordinal encoding, all fit on train
preprocessor = ColumnTransformer([
    ("num", Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("scale",  StandardScaler()),
    ]), numeric_cols),
    ("ord", OrdinalEncoder(
        categories=[["monthly", "annual", "two-year"]],
        handle_unknown="use_encoded_value",
        unknown_value=-1,
    ), ordinal_cols),
    ("nom", Pipeline([
        ("impute", SimpleImputer(strategy="most_frequent")),
        ("encode", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ]), nominal_cols),
])

model_corrected = Pipeline([
    ("prep", preprocessor),
    ("clf",  RandomForestClassifier(
        n_estimators=100, random_state=42, class_weight="balanced"
    )),
])

model_corrected.fit(X_train, y_train)
y_pred = model_corrected.predict(X_test)

print(classification_report(y_test, y_pred, target_names=["active", "churned"]))

### Compare the two pipelines

In [ ]:
from sklearn.metrics import accuracy_score

corrected_accuracy = accuracy_score(y_test, y_pred)

print(f"Broken pipeline accuracy:    {broken_accuracy:.1%}")
print(f"Corrected pipeline accuracy: {corrected_accuracy:.1%}")
print()
print("The broken pipeline's accuracy is driven by two leaky columns.")
print("At inference time on active customers, both columns would be 0 for everyone,")
print("and the model would predict 'not churned' for every customer — useless.")
print()
print("The corrected pipeline reports a lower number. That number is real.")

In [ ]:
# What features does the corrected model rely on?
feature_names = (
    numeric_cols
    + ordinal_cols
    + model_corrected.named_steps["prep"]
        .named_transformers_["nom"]
        .named_steps["encode"]
        .get_feature_names_out(nominal_cols)
        .tolist()
)

importances_c = pd.Series(
    model_corrected.named_steps["clf"].feature_importances_,
    index=feature_names
).sort_values(ascending=False)

print("Top features in corrected model:")
print(importances_c.round(3))

---
## Reflection

Write 3–5 bullet points summarising what this audit demonstrated about AI-generated
pipelines. Focus on principles that generalise beyond this specific dataset.

*(Write your reflection here.)*

- 
- 
- 